# IEEE Fraud Detection — XGBoost pipeline (clean)

Streamlined version of Chris Deotte's [Kaggle kernel](https://www.kaggle.com/cdeotte/xgb-fraud-with-magic-0-9600) (1st place solution).

**What this notebook does**
1. Load transaction + identity tables (curated columns from prior EDA).
2. Preprocess time-delta `D*` columns and encode categoricals.
3. Build baseline features → **XGB baseline** (~0.95 LB in original).
4. Add UID group aggregates ("magic") → **XGB + magic** (~0.96 LB in original).
5. Time-based **GroupKFold** cross-validation and optional submission file.

**Why this structure?** The original competition was won not by a exotic model but by **careful feature engineering on top of gradient boosting**. Fraud is extremely imbalanced (~3.5% positive rate), so the metric is ROC-AUC and the signal lives in subtle patterns: card/device/email combinations, time-since-last-transaction (`D*` columns), and how a transaction compares to that user's history. The notebook mirrors that two-stage recipe: a strong baseline, then "magic" UID aggregates that capture per-user behavior.

**Removed from the original** (noise / environment-specific): embedded images, leaderboard screenshots, `%%time`, verbose prints, duplicate `BUILD95`/`BUILD96` blocks, GPU-only settings, Kaggle-only post-processing UID files.

**Prerequisites:** `pip install -r ../requirements.txt` and IEEE data under `../data/` (see `README.md`).

## 1. Configuration

All tunables live in one place so you can smoke-test locally without waiting hours for 5,000 trees × 6 folds.

| Setting | Purpose |
|---------|---------|
| `FAST_MODE` | Fewer trees & folds for a quick smoke test |
| `RUN_BASELINE` | Train model without UID aggregates |
| `RUN_MAGIC` | Train model with UID group features |
| `USE_PARQUET` | Faster loads when parquet files exist |

**Intuition:** Full training in the original kernel takes a long time. `FAST_MODE` trades accuracy for speed (2 folds, 300 trees) so you can verify the pipeline runs end-to-end. `RUN_BASELINE` / `RUN_MAGIC` let you ablate the two model stages independently — useful when debugging feature code. Parquet avoids re-parsing 590k+ rows from CSV on every run.

### Code block: imports, paths, and XGB hyperparameters

**Imports**
- `Path` — portable file paths relative to the notebook (`../data`, `../outputs`).
- `gc` — explicit garbage collection after large merges/fits; the dataset is ~400+ columns × 590k rows and memory pressure is real on laptops.
- `GroupKFold` — CV that keeps entire calendar months together (see §7); plain KFold would leak future months into training folds.
- `warnings.filterwarnings` — suppresses noisy pandas FutureWarnings so fold logs stay readable.

**Paths & flags**
- `OUTPUT_DIR.mkdir(..., exist_ok=True)` — writes OOF predictions and submission CSVs without manual setup.
- `FAST_MODE` gates tree count, fold count, and early-stopping patience in one switch.

**XGB hyperparameters — why these values?**
| Param | Value | Rationale |
|-------|-------|-----------|
| `max_depth=12` | Deep enough to capture high-order interactions (card × email × amount) without the original kernel's GPU-specific tuning |
| `learning_rate=0.02` | Slow learning + many trees (5000) = better generalization on noisy fraud signal |
| `subsample=0.8`, `colsample_bytree=0.4` | Row/column bagging reduces overfit on rare fraud patterns |
| `missing=-1` | Pairs with our preprocessing: NaNs become -1, XGB treats -1 as the missing sentinel |
| `tree_method="hist"` | Histogram-based splits; fast on CPU, no GPU required |
| `eval_metric="auc"` | Matches competition metric; early stopping monitors validation AUC |

**`START_DATE`** — IEEE `TransactionDT` is seconds since an undisclosed epoch. Deotte discovered the anchor is **2017-11-30**; we use it to derive calendar month `DT_M` for time-based CV grouping.

In [ ]:
from pathlib import Path
import datetime
import gc
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = False          # True → 2 folds, 300 trees (smoke test)
RUN_BASELINE = True
RUN_MAGIC = True
USE_PARQUET = True         # fall back to CSV if parquet missing
N_SPLITS = 2 if FAST_MODE else 6
N_ESTIMATORS_CV = 300 if FAST_MODE else 5000
N_ESTIMATORS_LOCAL = 200 if FAST_MODE else 2000
EARLY_STOPPING = 50 if FAST_MODE else 200
VERBOSE_FIT = 50

XGB_PARAMS = dict(
    max_depth=12,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.4,
    missing=-1,
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
)

START_DATE = datetime.datetime(2017, 11, 30)


## 2. Column selection (from external EDA)

Prior work ([V/ID correlation EDA](https://www.kaggle.com/cdeotte/eda-for-columns-v-and-id)) dropped ~219 redundant `V*` columns.
We keep the same ~90 `V` indices and base transaction/identity fields.

**Why not use all 339 `V` columns?** Many `V*` features are near-duplicates (high pairwise correlation) or mostly missing. Tree models suffer from redundant/noisy columns — they slow training and can hurt generalization. The kept indices were chosen by correlation / importance analysis in the original EDA notebooks.

**`STR_COLS`** — categorical columns stored as pandas `category` dtype on load (memory-efficient, preserves levels).

**`BASE_COLS`** — core transaction fields: amount, card bins, address, email domains, count features `C*`, time deltas `D*`, match features `M*`.

**`V_KEEP`** — the surviving anonymized VFE (Vesta Feature Engineering) columns; these encode device/browser/network signals without revealing raw PII.

**`ID_COLS`** — identity-table columns joined later; includes both `id_12` and `id-12` naming variants because train vs test files use different conventions.

**`DTYPES`** — force `float32` for numerics (half the RAM of float64) and `category` for strings.

**`DROP_ALWAYS`** — columns removed before modeling:
- `TransactionDT` — raw timestamp; we derive `DT_M` / `day` instead, and keeping raw seconds adds little tree signal after preprocessing.
- Specific `D6–D14`, `C3`, `M5`, sparse `id_*` — dropped after time-consistency / sparsity EDA showed they hurt or added noise.
- `card4` — redundant with other card features after encoding.

**`DROP_MAGIC_ONLY`** — helper columns used to *build* features but not fed to the model (`uid`, `oof`, `DT_M`, `day`). Including them would leak grouping keys or target information.

### Code block: column lists and dtypes

This cell is **declarative configuration** — no computation, just the exact column manifest the rest of the pipeline expects.

- **`STR_COLS += [c.replace("_", "-") ...]`** — train identity uses underscores (`id_12`), test uses hyphens (`id-12`). We list both so either naming survives the merge/rename step.
- **`LOAD_COLS = BASE_COLS + [f"V{n}" for n in V_KEEP]`** — only these columns are read from disk (saves memory vs loading all 339 V columns).
- **`DROP_ALWAYS` / `DROP_MAGIC_ONLY`** — central allowlists for `feature_columns()` so train and test always use the same feature set without copy-pasting drop logic in multiple places.

In [ ]:
STR_COLS = [
    "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29", "id_30",
    "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38",
    "DeviceType", "DeviceInfo",
]
STR_COLS += [c.replace("_", "-") for c in STR_COLS if c.startswith("id_")]

BASE_COLS = [
    "TransactionID", "TransactionDT", "TransactionAmt",
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11",
    "C12", "C13", "C14", "D1", "D2", "D3", "D4", "D5", "D6", "D7", "D8",
    "D9", "D10", "D11", "D12", "D13", "D14", "D15", "M1", "M2", "M3", "M4",
    "M5", "M6", "M7", "M8", "M9",
]

V_KEEP = (
    [1, 3, 4, 6, 8, 11]
    + [13, 14, 17, 20, 23, 26, 27, 30]
    + [36, 37, 40, 41, 44, 47, 48]
    + [54, 56, 59, 62, 65, 67, 68, 70]
    + [76, 78, 80, 82, 86, 88, 89, 91]
    + [107, 108, 111, 115, 117, 120, 121, 123]
    + [124, 127, 129, 130, 136]
    + [138, 139, 142, 147, 156, 162]
    + [165, 160, 166]
    + [178, 176, 173, 182]
    + [187, 203, 205, 207, 215]
    + [169, 171, 175, 180, 185, 188, 198, 210, 209]
    + [218, 223, 224, 226, 228, 229, 235]
    + [240, 258, 257, 253, 252, 260, 261]
    + [264, 266, 267, 274, 277]
    + [220, 221, 234, 238, 250, 271]
    + [294, 284, 285, 286, 291, 297]
    + [303, 305, 307, 309, 310, 320]
    + [281, 283, 289, 296, 301, 314]
)

LOAD_COLS = BASE_COLS + [f"V{n}" for n in V_KEEP]

ID_COLS = (
    [f"id_0{x}" for x in range(1, 10)]
    + [f"id_{x}" for x in range(10, 34)]
    + [f"id-0{x}" for x in range(1, 10)]
    + [f"id-{x}" for x in range(10, 34)]
)

DTYPES = {c: "float32" for c in LOAD_COLS + ID_COLS}
for c in STR_COLS:
    DTYPES[c] = "category"

# Features dropped after time-consistency / sparsity EDA
DROP_ALWAYS = (
    ["TransactionDT"]
    + [f"D{i}" for i in [6, 7, 8, 9, 12, 13, 14]]
    + ["C3", "M5", "id_08", "id_33"]
    + ["card4", "id_07", "id_14", "id_21", "id_30", "id_32", "id_34"]
    + [f"id_{x}" for x in range(22, 28)]
)

DROP_MAGIC_ONLY = ["oof", "DT_M", "day", "uid"]


## 3. Load and merge data

Train: transaction + identity on `TransactionID`.  
Test identity columns are renamed to match train (`id-12` → `id_12`, etc.).

### Code block: `_read_table` and `load_datasets`

**Why left-merge identity?** Not every transaction has identity info (~25% missing). A left join keeps all transactions; missing identity becomes NaN/-1 after preprocessing.

**Why `set_index("TransactionID")`?** Stable row key for merges and for writing submission/OOF files back to the correct IDs.

**Parquet vs CSV logic**
1. Prefer parquet when `USE_PARQUET=True` — column pruning still applies via `[usecols]`.
2. Fall back to CSV with explicit `dtype=DTYPES` so categoricals and float32 load correctly in one pass.
3. Raise early if columns are missing — catches stale parquet exports after EDA changes.

**Test identity rename** — without this, train columns (`id_12`) and test columns (`id-12`) would be separate features and the model would never see test identity signal.

**`del` + `gc.collect()`** — identity tables are large; free them immediately after merge since only the combined frame is needed.

**Fraud rate print** — sanity check (~3.5% expected); if you see something wildly different, the label column or merge went wrong.

In [ ]:
def _read_table(stem: str, usecols):
    pq = DATA_DIR / f"{stem}.parquet"
    csv = DATA_DIR / f"{stem}.csv"
    if USE_PARQUET and pq.exists():
        df = pd.read_parquet(pq)
        missing = [c for c in usecols if c not in df.columns]
        if missing:
            raise KeyError(f"{stem}: missing columns {missing[:5]}")
        df = df[usecols]
    elif csv.exists():
        df = pd.read_csv(csv, usecols=usecols, dtype=DTYPES)
    else:
        raise FileNotFoundError(f"Need {pq} or {csv}")
    if "TransactionID" in df.columns:
        df = df.set_index("TransactionID")
    return df


def load_datasets():
    tx_train_cols = LOAD_COLS + ["isFraud"]
    X_train = _read_table("train_transaction", tx_train_cols)
    train_id = _read_table("train_identity", ID_COLS)
    X_train = X_train.merge(train_id, how="left", left_index=True, right_index=True)

    X_test = _read_table("test_transaction", LOAD_COLS)
    test_id = _read_table("test_identity", ID_COLS)
    rename = dict(zip(test_id.columns, train_id.columns))
    test_id = test_id.rename(columns=rename)
    X_test = X_test.merge(test_id, how="left", left_index=True, right_index=True)

    y = X_train["isFraud"].copy()
    X_train = X_train.drop(columns=["isFraud"])
    del train_id, test_id
    gc.collect()
    return X_train, X_test, y


X_train, X_test, y_train = load_datasets()
print(f"Train {X_train.shape} | Test {X_test.shape} | Fraud rate {y_train.mean():.4f}")


## 4. Preprocess

**D-columns:** convert time deltas to a stable time anchor:  
`D_i ← D_i - TransactionDT / 86400` (skip D1, D2, D3, D5, D9).

**All other numerics:** shift to non-negative, NaN → -1 (XGB `missing=-1`).  
**Categoricals:** joint label-encoding (train + test).

### Code block: preprocessing loop — intuition

**D-column normalization (the key insight)**  
Raw `D*` values are *days since some past event* (last transaction, password change, etc.) measured at transaction time. Because `TransactionDT` drifts over the 3-month dataset, the same absolute `D4` value can mean different things in January vs March. Subtracting `TransactionDT / 86400` re-anchors each delta to a **fixed calendar reference**, making patterns stable across time. We skip D1, D2, D3, D5, D9 because those are used differently downstream (D1 defines UID day offset; others were kept raw in the winning solution).

**Categorical factorization on train+test together**  
Every category seen in test must get an integer code the model can recognize. Fitting encoders on train-only would map unseen test categories to NaN. `sort=True` makes codes reproducible. `int16` vs `int32` is a memory choice when cardinality exceeds 32k.

**Numeric shift + fillna(-1)**  
XGBoost's `missing=-1` requires a sentinel distinct from valid values. Shifting by the global minimum ensures real values are ≥ 0 and -1 unambiguously means "missing". We skip `TransactionAmt` and `TransactionDT` because amount keeps its natural scale (trees split on round dollars + `cents` feature) and DT is dropped anyway.

In [ ]:
SECONDS_PER_DAY = np.float32(24 * 60 * 60)
SKIP_D_NORMALIZE = {1, 2, 3, 5, 9}

for i in range(1, 16):
    if i in SKIP_D_NORMALIZE:
        continue
    col = f"D{i}"
    X_train[col] = X_train[col] - X_train.TransactionDT / SECONDS_PER_DAY
    X_test[col] = X_test[col] - X_test.TransactionDT / SECONDS_PER_DAY

for col in X_train.columns:
    if str(X_train[col].dtype) in ("category", "object"):
        combined, _ = pd.concat([X_train[col], X_test[col]]).factorize(sort=True)
        dtype = "int32" if combined.max() > 32000 else "int16"
        n = len(X_train)
        X_train[col] = combined[:n].astype(dtype)
        X_test[col] = combined[n:].astype(dtype)
    elif col not in ("TransactionAmt", "TransactionDT"):
        mn = float(min(X_train[col].min(), X_test[col].min()))
        X_train[col] = (X_train[col] - mn).fillna(-1)
        X_test[col] = (X_test[col] - mn).fillna(-1)

gc.collect()


## 5. Feature engineering helpers

Reusable encoding primitives from the original kernel. Each computes statistics on **train + test combined** so test rows get the same mappings as train (competition-style; see §11 for production caveats).

| Function | Description |
|----------|-------------|
| `encode_fe` | Frequency encoding (train+test) |
| `encode_le` | Label encoding for combined categoricals |
| `encode_ag` | Group mean/std |
| `encode_cb` | Concatenate two columns then label-encode |
| `encode_ag2` | Count of unique values per group |

### Code block: helper functions — line-by-line intuition

**`encode_fe` (frequency encoding)**  
Maps each category to its relative frequency in the combined dataset. Rare categories → low values, common ones → high. Fraudsters often use unusual card/address combos; frequency captures "how common is this value?" without exploding dimensionality like one-hot. `mapping[-1] = -1` preserves the missing sentinel.

**`encode_le` (label encoding)**  
Integer codes for high-cardinality strings created by `encode_cb`. Same train+test factorize logic as preprocessing.

**`encode_ag` (group aggregates)**  
The workhorse for "magic" features. For each `(main_col, group_col, agg)` triple, compute e.g. mean transaction amount **per group** and map back to every row. Intuition: "Is this transaction amount unusual for this card?" — a classic fraud signal. `use_na=True` treats -1 as NaN before aggregating so missing values don't drag means down.

**`encode_cb` (concatenate + label encode)**  
Builds interaction keys like `card1_addr1` = `"12345_678"`. Trees can learn crosses implicitly, but explicit concatenated keys make **group-by** aggregations possible on compound identities.

**`encode_ag2` (unique count per group)**  
Counts how many distinct values of `main` appear within each `group`. Example: if a UID uses 5 different email domains, that's suspicious. Complements mean/std aggregates with diversity signals.

In [ ]:
def encode_fe(df_train, df_test, columns):
    for col in columns:
        freq = pd.concat([df_train[col], df_test[col]]).value_counts(normalize=True, dropna=True)
        mapping = freq.to_dict()
        mapping[-1] = -1
        name = f"{col}_FE"
        df_train[name] = df_train[col].map(mapping).astype("float32")
        df_test[name] = df_test[col].map(mapping).astype("float32")


def encode_le(df_train, df_test, col):
    combined, _ = pd.concat([df_train[col], df_test[col]]).factorize(sort=True)
    dtype = "int32" if combined.max() > 32000 else "int16"
    n = len(df_train)
    df_train[col] = combined[:n].astype(dtype)
    df_test[col] = combined[n:].astype(dtype)


def encode_ag(df_train, df_test, main_cols, group_cols, aggs=("mean",), use_na=False):
    for main in main_cols:
        for grp in group_cols:
            for agg in aggs:
                name = f"{main}_{grp}_{agg}"
                part = pd.concat([df_train[[grp, main]], df_test[[grp, main]]])
                if use_na:
                    part.loc[part[main] == -1, main] = np.nan
                stats = part.groupby(grp)[main].agg(agg)
                df_train[name] = df_train[grp].map(stats).astype("float32").fillna(-1)
                df_test[name] = df_test[grp].map(stats).astype("float32").fillna(-1)


def encode_cb(df_train, df_test, col1, col2):
    name = f"{col1}_{col2}"
    df_train[name] = df_train[col1].astype(str) + "_" + df_train[col2].astype(str)
    df_test[name] = df_test[col1].astype(str) + "_" + df_test[col2].astype(str)
    encode_le(df_train, df_test, name)


def encode_ag2(df_train, df_test, main_cols, group_cols):
    for main in main_cols:
        for grp in group_cols:
            name = f"{grp}_{main}_ct"
            part = pd.concat([df_train[[grp, main]], df_test[[grp, main]]])
            counts = part.groupby(grp)[main].nunique()
            df_train[name] = df_train[grp].map(counts).astype("float32").fillna(-1)
            df_test[name] = df_test[grp].map(counts).astype("float32").fillna(-1)


## 6. Baseline features (no UID)

Engineered features that passed local validation in the original kernel:
- `cents`, frequency encodings, `card1_addr1` keys, group stats on amount / D9 / D11.

### Code block: `build_baseline_features` — why each feature?

**`cents`** — fractional part of `TransactionAmt` (e.g. $10.99 → 0.99). Fraud rings sometimes use distinctive cent patterns; separating dollars from cents gives trees an easy split.

**Frequency encodings on `addr1`, `card1`, `card2`, `card3`, `P_emaildomain`** — how common is each raw value? Rare addresses/cards correlate with fraud.

**`encode_cb("card1", "addr1")` → `card1_addr1`** — compound identity: same card at same address ≈ same household. Stronger grouping key than card alone.

**`encode_cb("card1_addr1", "P_emaildomain")`** — adds email to the identity; email domains differ widely between legit users and fraud farms.

**Second-round FE on the compound keys** — frequency of the *interaction* itself (not just individual columns).

**Group aggregates on `TransactionAmt`, `D9`, `D11` by card / card_addr1 / card_addr1_email**  
- `TransactionAmt` mean/std → "is this amount typical for this card?"
- `D9`, `D11` → time-since-event features; group stats capture whether this transaction's timing is normal for that identity.

**`add_month` → `DT_M`** — calendar month index used later as GroupKFold groups (not a model feature in baseline; listed in `DROP_MAGIC_ONLY` when magic runs, but needed for CV throughout).

**`feature_columns(include_magic=False)`** — returns all columns minus `DROP_ALWAYS`. Baseline uses ~216 features in the original kernel.

In [ ]:
def add_month(df):
    dt = START_DATE + pd.to_timedelta(df.TransactionDT, unit="s")
    df["DT_M"] = (dt.dt.year - 2017) * 12 + dt.dt.month


def build_baseline_features(df_train, df_test):
    df_train["cents"] = (df_train.TransactionAmt - np.floor(df_train.TransactionAmt)).astype("float32")
    df_test["cents"] = (df_test.TransactionAmt - np.floor(df_test.TransactionAmt)).astype("float32")

    encode_fe(df_train, df_test, ["addr1", "card1", "card2", "card3", "P_emaildomain"])
    encode_cb(df_train, df_test, "card1", "addr1")
    encode_cb(df_train, df_test, "card1_addr1", "P_emaildomain")
    encode_fe(df_train, df_test, ["card1_addr1", "card1_addr1_P_emaildomain"])
    encode_ag(
        df_train, df_test,
        ["TransactionAmt", "D9", "D11"],
        ["card1", "card1_addr1", "card1_addr1_P_emaildomain"],
        aggs=["mean", "std"],
        use_na=True,
    )
    add_month(df_train)
    add_month(df_test)


build_baseline_features(X_train, X_test)


def feature_columns(include_magic=False):
    drop = list(DROP_ALWAYS)
    if include_magic:
        drop = drop + list(DROP_MAGIC_ONLY)
    return [c for c in X_train.columns if c not in drop]


baseline_cols = feature_columns(include_magic=False)
print(f"Baseline feature count: {len(baseline_cols)}")


## 7. Training utilities

- **Local validation:** first 75% of rows by time index → train, last 25% → validate.
- **OOF CV:** `GroupKFold` on calendar month `DT_M` (6 folds in full run).

### Code block: training functions — validation strategy

**Why time-based validation?** The dataset spans Dec 2017 – Mar 2018. Random splits leak future fraud patterns into training and inflate AUC. The competition test set is later in time, so validation must respect temporal order.

**`local_time_split`** — simple 75/25 chronological cut on row order (data is sorted by `TransactionDT` in the source files). Used only in `FAST_MODE` for a quick sanity check.

**`train_group_kfold` — the real evaluation loop**
- **`GroupKFold(groups=df["DT_M"])`** — each fold holds out one entire calendar month. All rows from the same month stay together, preventing within-month leakage across train/valid boundary.
- **Fresh model per fold** — no information from validation months enters training.
- **`early_stopping_rounds`** — stops adding trees when validation AUC plateaus; with 5000 max trees and lr=0.02, typically hundreds–thousands of trees actually fit.
- **OOF predictions** — each row gets a prediction from the fold where it was validation data. Stacked OOF AUC approximates true generalization.
- **Test predictions averaged across folds** — standard ensembling; reduces variance vs a single model.
- **CSV outputs** — `oof_*.csv` for analysis/blending; `sub_*.csv` for submission format.

**`quick_local_fit`** — lighter alternative: one time split, fewer trees (`N_ESTIMATORS_LOCAL`). Good for debugging feature code without 6× training cost.

In [ ]:
def local_time_split(df):
    n = len(df)
    cut = 3 * n // 4
    idx = df.index
    return idx[:cut], idx[cut:]


def train_group_kfold(df, y, feature_cols, label, save_oof=True, save_preds=True):
    oof = np.zeros(len(df))
    test_pred = np.zeros(len(X_test))

    gkf = GroupKFold(n_splits=N_SPLITS)
    for fold, (idx_train, idx_valid) in enumerate(gkf.split(df, y, groups=df["DT_M"])):
        month = df.iloc[idx_valid]["DT_M"].iloc[0]
        print(f"[{label}] fold {fold} — holdout month {month} | train {len(idx_train):,} | valid {len(idx_valid):,}")

        model = xgb.XGBClassifier(n_estimators=N_ESTIMATORS_CV, **XGB_PARAMS)
        model.fit(
            df.iloc[idx_train][feature_cols],
            y.iloc[idx_train],
            eval_set=[(df.iloc[idx_valid][feature_cols], y.iloc[idx_valid])],
            verbose=VERBOSE_FIT,
            early_stopping_rounds=EARLY_STOPPING,
        )
        oof[idx_valid] = model.predict_proba(df.iloc[idx_valid][feature_cols])[:, 1]
        test_pred += model.predict_proba(X_test[feature_cols])[:, 1] / N_SPLITS
        del model
        gc.collect()

    auc = roc_auc_score(y, oof)
    print(f"[{label}] OOF ROC-AUC: {auc:.5f}")

    if save_oof:
        pd.DataFrame({"TransactionID": df.index, "oof": oof}).to_csv(
            OUTPUT_DIR / f"oof_{label}.csv", index=False
        )
    if save_preds:
        sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
        sub["isFraud"] = test_pred
        sub.to_csv(OUTPUT_DIR / f"sub_{label}.csv", index=False)
        print(f"Wrote {OUTPUT_DIR / f'sub_{label}.csv'}")

    return oof, test_pred, auc


def quick_local_fit(df, y, feature_cols, label):
    idx_train, idx_valid = local_time_split(df)
    model = xgb.XGBClassifier(n_estimators=N_ESTIMATORS_LOCAL, **XGB_PARAMS)
    model.fit(
        df.loc[idx_train, feature_cols],
        y.loc[idx_train],
        eval_set=[(df.loc[idx_valid, feature_cols], y.loc[idx_valid])],
        verbose=VERBOSE_FIT,
        early_stopping_rounds=min(100, EARLY_STOPPING),
    )
    valid_pred = model.predict_proba(df.loc[idx_valid, feature_cols])[:, 1]
    auc = roc_auc_score(y.loc[idx_valid], valid_pred)
    print(f"[{label}] local holdout ROC-AUC: {auc:.5f}")
    return model, auc


## 8. Baseline model (XGB without magic)

Target: ~0.95 public LB in the original competition with ~216 features.

### Code block: baseline training

**`if RUN_BASELINE`** — gate so you can skip re-training when iterating on magic features only.

**`FAST_MODE` branch** — calls `quick_local_fit` instead of full GroupKFold. Expect lower, noisier AUC but fast feedback.

**Full run branch** — `train_group_kfold` with 6 monthly folds and 5000 trees. This reproduces the "0.95" stage from the original kernel (`BUILD95` in Deotte's notebook).

**Why train baseline first?** Establishes a reference AUC before adding magic features. In the original solution, magic added ~0.01 AUC on top of baseline — if baseline is broken, magic gains are meaningless.

In [ ]:
baseline_auc = None
if RUN_BASELINE:
    if FAST_MODE:
        _, baseline_auc = quick_local_fit(X_train, y_train, baseline_cols, "xgb_baseline")
    else:
        _, _, baseline_auc = train_group_kfold(X_train, y_train, baseline_cols, "xgb_baseline")
else:
    print("Skipped baseline training (RUN_BASELINE=False)")


## 9. Magic features (UID aggregates)

**UID** approximates a cardholder: `card1_addr1 + floor(day - D1)`.

Group statistics are computed on train+test, then **UID is excluded** from model features (only aggregates are used).

### Code block: `build_magic_features` — the "magic" explained

This is the ~0.01 AUC lift that took the solution from 0.95 → 0.96 on the Kaggle leaderboard.

**Why UID?**  
`card1_addr1` identifies a payment instrument + billing address, but the same card can be used by different people over time, or a fraudster may reuse a stolen card across days. **`D1`** is "days since first transaction on this card" — subtracting it from calendar `day` gives an approximate **session/day index** for that cardholder. Concatenating:

```
uid = str(card1_addr1) + "_" + str(floor(day - D1))
```

…creates a pseudo-user ID that clusters transactions by the same person using the same card at the same address on the same "card-age day". It's not perfect PII, but it groups behavior well enough for aggregate features.

**`encode_fe(["uid"])`** — how often does this pseudo-user appear? One-off UIDs vs repeat customers.

**Group mean/std on amount and D-columns by UID**  
Compares each transaction to *that user's* typical amount and timing. Fraud often looks like: "this user's 50th transaction is 10× their usual amount" or "D15 is unusual for this UID".

**Group mean on `C1–C14` (except C3) and `M1–M9` by UID**  
Count/match features averaged per user — captures whether this transaction's device/browser signals match the user's history.

**`encode_ag2` on email, dist1, month, id_02, cents, C13, V314, V127, …**  
Diversity features: fraud rings rotate emails, devices, or cent patterns within one UID. High unique counts = suspicious.

**`encode_ag` std on `C14` by UID** — variability of a specific count feature within user.

**`outsider15`** — binary flag: `|D1 - D15| > 3`. If D15 (days since something) diverges heavily from D1 (days since first txn), the transaction may belong to a different "session" — a hand-crafted rule from EDA that survived validation.

**UID excluded from model** — raw UID has extreme cardinality (millions of levels). Trees can't generalize on the ID itself; only the **aggregated statistics** carry signal. That's why `uid` is in `DROP_MAGIC_ONLY`.

In [ ]:
def build_magic_features(df_train, df_test):
    df_train["day"] = df_train.TransactionDT / SECONDS_PER_DAY
    df_test["day"] = df_test.TransactionDT / SECONDS_PER_DAY
    df_train["uid"] = (
        df_train.card1_addr1.astype(str) + "_" + np.floor(df_train.day - df_train.D1).astype(str)
    )
    df_test["uid"] = (
        df_test.card1_addr1.astype(str) + "_" + np.floor(df_test.day - df_test.D1).astype(str)
    )

    encode_fe(df_train, df_test, ["uid"])
    encode_ag(df_train, df_test, ["TransactionAmt", "D4", "D9", "D10", "D15"], ["uid"], ["mean", "std"], use_na=True)
    encode_ag(df_train, df_test, [f"C{i}" for i in range(1, 15) if i != 3], ["uid"], ["mean"], use_na=True)
    encode_ag(df_train, df_test, [f"M{i}" for i in range(1, 10)], ["uid"], ["mean"], use_na=True)
    encode_ag2(df_train, df_test, ["P_emaildomain", "dist1", "DT_M", "id_02", "cents"], ["uid"])
    encode_ag(df_train, df_test, ["C14"], ["uid"], ["std"], use_na=True)
    encode_ag2(df_train, df_test, ["C13", "V314"], ["uid"])
    encode_ag2(df_train, df_test, ["V127", "V136", "V309", "V307", "V320"], ["uid"])

    df_train["outsider15"] = (np.abs(df_train.D1 - df_train.D15) > 3).astype("int8")
    df_test["outsider15"] = (np.abs(df_test.D1 - df_test.D15) > 3).astype("int8")


if RUN_MAGIC:
    build_magic_features(X_train, X_test)
    magic_cols = feature_columns(include_magic=True)
    print(f"Magic feature count: {len(magic_cols)}")
else:
    magic_cols = []
    print("Skipped magic features (RUN_MAGIC=False)")


## 10. Magic model (XGB + UID aggregates)

Adds ~47 group features; original kernel gained ~0.01 local AUC vs baseline.

### Code block: magic model training

Same training harness as baseline (`train_group_kfold` or `quick_local_fit`) but with `magic_cols = feature_columns(include_magic=True)`.

**Feature set difference** — `include_magic=True` additionally drops `DROP_MAGIC_ONLY` columns (`uid`, `day`, `DT_M`, `oof`) so the model sees raw + baseline + magic aggregate columns (~263 features in original).

**`if RUN_MAGIC`** — allows running baseline-only for comparison or debugging.

**Expected outcome** — magic OOF AUC should exceed baseline OOF AUC by roughly 0.005–0.015 if the pipeline matches the original kernel. Smaller gaps can happen on different hardware/sample or without post-processing.

In [ ]:
magic_auc = None
if RUN_MAGIC:
    if FAST_MODE:
        _, magic_auc = quick_local_fit(X_train, y_train, magic_cols, "xgb_magic")
    else:
        _, _, magic_auc = train_group_kfold(X_train, y_train, magic_cols, "xgb_magic")
else:
    print("Skipped magic training")


## 11. Summary

Original kernel also applied **post-processing** with external UID files (not reproduced here — needs Kaggle-only assets).

For production, prefer **train-only** group aggregates or target encoding with proper CV to avoid leakage.

### Code block: results summary

Collects run metadata into a single table for quick comparison:

| Key | Meaning |
|-----|---------|
| `fast_mode` / `n_splits` | Which validation regime ran |
| `baseline_features` / `magic_features` | Feature counts after drop lists |
| `baseline_oof_auc` / `magic_oof_auc` | Out-of-fold ROC-AUC from GroupKFold (or local holdout in FAST_MODE) |

**How to read the AUCs** — OOF AUC is the honest estimate of generalization. Compare baseline vs magic to quantify the value of UID aggregates. In FAST_MODE, numbers are indicative only (2 folds, fewer trees).

**Production warning (important)** — this notebook computes group stats on **train + test combined**, which is standard for Kaggle (test has no labels, so it's not label leakage) but **would leak in production** if test distributions differ. In deployment, compute aggregates on training/historical data only, or use out-of-fold target encoding within CV folds.

In [ ]:
summary = {
    "fast_mode": FAST_MODE,
    "n_splits": N_SPLITS,
    "baseline_features": len(baseline_cols) if RUN_BASELINE else 0,
    "magic_features": len(magic_cols) if RUN_MAGIC else 0,
    "baseline_oof_auc": baseline_auc,
    "magic_oof_auc": magic_auc,
}
pd.Series(summary).to_frame("value")
